# TESDA learner matcher · v6 (every related qualification)

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Training takes roughly an hour on a T4.
Each cell stands on its own (settings are saved to disk), so after a session restart just choose Run all again.

v6 continues the published v4 model. Each training query is paired with **all** the qualifications it belongs to
(a job, Core skill or keyword several qualifications share keeps every owner), and is scored against the **whole
catalog**: every related qualification must outrank every unrelated one. Other levels of the same family are left
neutral unless the query names a level. The data adds 5,736 hand-written Filipino and Taglish
drafts (jobs, everyday tasks, short searches, typos, experience and indirect goals) for all 319 qualifications.
So that matching more does not cost rejection, every batch also carries goals that must get **no** match
(598: careers with no TESDA qualification, non-goals, vague requests), and their best
z-score against the catalog, the number the app rejects on, is pushed below its floor.

Everything is embedded here: 56,592 training queries and the public catalog passages. Evaluation goals
and MVP probes were removed before building it. With **USE_DRIVE** on, checkpoints go to
`MyDrive/tesda-track-training/` every 100 steps; after a disconnect, run all again and it resumes.
The last cell downloads `tesda-e5-small-<ROUND>.zip` (ROUND defaults to `v6-colab`). Calibrate and evaluate it locally (see `backend/training/V6.md`).

In [ ]:
#@title 1. Install the pinned libraries (Colab's PyTorch) and require a GPU
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'sentence-transformers==6.1.0', 'transformers==5.19.0', 'accelerate==1.15.0', 'onnx==1.23.2', 'onnxruntime==1.31.0', 'fastembed==0.8.1'], check=True)
import torch
assert torch.cuda.is_available(), "Choose Runtime > Change runtime type > T4 GPU, then run all again"
print(torch.__version__, torch.cuda.get_device_name(0))
print("If Colab asks to restart the session, restart it and choose Runtime > Run all again.")


In [ ]:
#@title 2. Settings
ROUND = "v6-colab"             # keep it to resume a run; use a new name (v6-colab-2, ...) for a new experiment
EPOCHS = 1.0
LR = 1e-5
BATCH_SIZE = 48
UNRELATED_PER_BATCH = 8        # goals that must get no match, added to every batch
CHUNK = 16                     # texts per forward pass; 8 if the GPU runs out of memory (does not change results)
TRAIN_WORD_EMBEDDINGS = False  # True also updates the 250k-token vocabulary table (an experiment, not the default)
USE_DRIVE = True               # checkpoints survive a disconnect
SMOKE = False                  # True: 30 steps only, to check the export before a full run

import json, re
from pathlib import Path
assert re.fullmatch(r"v6(?:-[A-Za-z0-9_-]+)?", ROUND), "Use v6 or v6-your-run-name"
WORK = Path("/content/tesda-v6"); WORK.mkdir(exist_ok=True)
# Saved to disk so a session restart between cells cannot lose them.
(WORK / "settings.json").write_text(json.dumps(dict(
    ROUND=ROUND, EPOCHS=EPOCHS, LR=LR, BATCH_SIZE=BATCH_SIZE, UNRELATED_PER_BATCH=UNRELATED_PER_BATCH, CHUNK=CHUNK,
    TRAIN_WORD_EMBEDDINGS=TRAIN_WORD_EMBEDDINGS, USE_DRIVE=USE_DRIVE, SMOKE=SMOKE), indent=2))
print((WORK / "settings.json").read_text())


In [ ]:
#@title 3. Write the embedded training data and code
from pathlib import Path
import base64, gzip, hashlib
WORK = Path("/content/tesda-v6"); WORK.mkdir(exist_ok=True)
(WORK / "v6-dataset.json").write_bytes(gzip.decompress(base64.b64decode("...")))
(WORK / "v6_train.py").write_text('"""Standalone v6 trainer: multi-positive queries against the whole qualification catalog.\n\nNeeds only torch, sentence-transformers, onnx and fastembed plus a dataset written by `python -m training.v6\n--dataset FILE`, so the same file runs locally and in Google Colab (colab/train_colab_v6.ipynb embeds it).\n\nEach step scores a batch of learner queries against all qualification passages: a bank of passage vectors that is\nrefreshed during training, with gradients through the batch\'s positives and its hardest negatives (GradCache style,\nso memory stays bounded). Every positive must beat the log-sum-exp of all negatives; other levels of an owner\'s\nfamily are left out of the negatives unless the query names a level. Unrelated goals (no owner: careers with no TESDA\nqualification, non-goals, vague requests) are mixed into every batch, and their highest z-score against the catalog,\nthe number the app rejects on, is pushed below a margin, so matching more does not cost rejection.\n\n    python v6_train.py --data v6-dataset.json --base <recovered v4 folder> --output <new model folder>\n"""\nimport argparse\nimport hashlib\nimport json\nimport math\nimport random\nimport shutil\nimport time\nfrom datetime import datetime, timezone\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport numpy as np\n\nPARITY_TEXTS = 10\nMIN_PARITY = 0.999\n\n\ndef digest(path: Path) -> str:\n    with path.open("rb") as f:\n        return hashlib.file_digest(f, "sha256").hexdigest()\n\n\ndef load_dataset(path: Path):\n    data = json.loads(path.read_text(encoding="utf-8"))\n    names, families = data["names"], data["families"]\n    if len(names) != len(data["passages"]) or len(names) != len(families):\n        raise ValueError("Dataset names, passages and families differ in length")\n    by_family: dict[str, list[int]] = {}\n    for index, fam in enumerate(families):\n        by_family.setdefault(fam, []).append(index)\n    texts = [row[0] for row in data["items"]]\n    positives = np.zeros((len(texts), len(names)), dtype=bool)\n    neutral = np.zeros_like(positives)\n    held = np.array([bool(row[3]) for row in data["items"]])\n    for i, (_, owners, names_level, _) in enumerate(data["items"]):\n        if not names_level:\n            for owner in owners:\n                neutral[i, by_family[families[owner]]] = True\n        positives[i, owners] = True\n    neutral &= ~positives\n    return data, texts, positives, neutral, held\n\n\nclass Trainer:\n    def __init__(self, model, passages: list[str], chunk: int, scale: float, device: str, precision: str = "auto"):\n        import torch\n\n        self.torch, self.model, self.chunk, self.scale, self.device = torch, model, chunk, scale, device\n        self.passages = [f"passage: {p}" for p in passages]\n        if precision == "auto":\n            # Ampere or newer: bfloat16. Older GPUs such as Colab\'s T4: float16 with a gradient scaler.\n            native_bf16 = device == "cuda" and torch.cuda.get_device_capability()[0] >= 8\n            precision = "bf16" if native_bf16 else "fp16" if device == "cuda" else "fp32"\n        self.dtype = {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": torch.float32}[precision]\n        self.scaler = torch.amp.GradScaler("cuda", enabled=self.dtype == torch.float16)\n        self.bank = None\n\n    def embed(self, texts: list[str]):\n        torch = self.torch\n        prepare = getattr(self.model, "preprocess", None) or self.model.tokenize\n        features = {k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in prepare(texts).items()}\n        with torch.autocast(device_type=self.device, dtype=self.dtype, enabled=self.dtype != torch.float32):\n            output = self.model(features)["sentence_embedding"]\n        return output.float()\n\n    def embed_no_grad(self, texts: list[str]):\n        with self.torch.no_grad():\n            return self.torch.cat([self.embed(texts[i:i + self.chunk]) for i in range(0, len(texts), self.chunk)])\n\n    def refresh(self):\n        self.bank = self.embed_no_grad(self.passages)\n\n    def loss(self, queries, passages, pos, neu, unrelated_weight: float = 1.0, z_margin: float = 3.0):\n        """Mean over the batch\'s rows. A related row (one with owners): every positive must beat the log-sum-exp of\n        all negatives, neutral passages left out. An unrelated row: its soft-maximum z-score against the whole\n        catalog must stay below z_margin (the app shows nothing below its calibrated floor, about 3.5)."""\n        torch = self.torch\n        functional = torch.nn.functional\n        scores = self.scale * queries @ passages.T\n        related = pos.any(1)\n        per_row = torch.zeros(len(queries), device=queries.device)\n        if related.any():\n            s, p, n = scores[related], pos[related], neu[related]\n            lse = torch.logsumexp(s.masked_fill(p | n, float("-inf")), dim=1, keepdim=True)\n            per_row[related] = (functional.softplus(lse - s) * p).sum(1) / p.sum(1)\n        if (~related).any():\n            per_row[~related] = unrelated_weight * functional.softplus(2 * (self.top_z(scores[~related]) - z_margin)) / 2\n        return per_row.mean()\n\n    @staticmethod\n    def top_z(scores, sharpness: float = 4.0):\n        """A smooth maximum of each row\'s z-scores: how far the best qualification stands above the catalog."""\n        z = (scores - scores.mean(1, keepdim=True)) / scores.std(1, keepdim=True).clamp_min(1e-6)\n        return z.mul(sharpness).logsumexp(1) / sharpness\n\n    def step(self, texts: list[str], pos, neu, hard_k: int, max_passages: int, unrelated_weight: float = 1.0):\n        torch = self.torch\n        query_texts = [f"query: {t}" for t in texts]\n        queries = self.embed_no_grad(query_texts)\n        with torch.no_grad():\n            hard = (queries @ self.bank.T).masked_fill(pos | neu, float("-inf")).topk(hard_k, dim=1).indices.flatten()\n            chosen = list(dict.fromkeys(pos.nonzero()[:, 1].tolist() + hard.tolist()))[:max_passages]\n        index = torch.tensor(chosen, device=self.device)\n        fresh = self.embed_no_grad([self.passages[i] for i in chosen])\n        self.bank[index] = fresh\n        q_leaf = queries.clone().requires_grad_()\n        p_leaf = fresh.clone().requires_grad_()\n        loss = self.loss(q_leaf, self.bank.index_copy(0, index, p_leaf), pos, neu, unrelated_weight)\n        loss.backward()\n        # Second pass with gradients, chunk by chunk (no dropout: the model is in eval mode, so both passes agree).\n        # scaler.scale() multiplies by the loss scale under fp16 and returns the gradient unchanged otherwise.\n        for texts_, grads in ((query_texts, q_leaf.grad), ([self.passages[i] for i in chosen], p_leaf.grad)):\n            for start in range(0, len(texts_), self.chunk):\n                self.embed(texts_[start:start + self.chunk]).backward(\n                    self.scaler.scale(grads[start:start + self.chunk]))\n        return loss.item(), len(chosen)\n\n    def evaluate(self, texts, pos, neu, batch: int = 128) -> tuple[float, float | None]:\n        """Mean loss, and the share of unrelated rows whose top z-score is below 3.5 (a rough rejection proxy)."""\n        total, below, unrelated = 0.0, 0, 0\n        with self.torch.no_grad():\n            for start in range(0, len(texts), batch):\n                q = self.embed_no_grad([f"query: {t}" for t in texts[start:start + batch]])\n                p, n = pos[start:start + batch], neu[start:start + batch]\n                total += float(self.loss(q, self.bank, p, n)) * len(q)\n                none = ~p.any(1)\n                if none.any():\n                    z = self.top_z(self.scale * q[none] @ self.bank.T, sharpness=50.0)\n                    below += int((z < 3.5).sum())\n                    unrelated += int(none.sum())\n        return total / max(len(texts), 1), (below / unrelated if unrelated else None)\n\n\ndef export_onnx(folder: Path) -> Path:\n    """The transformer as ONNX with the inputs and output fastembed runs (same export as training/finetune.py)."""\n    import torch\n    from transformers import AutoModel, AutoTokenizer\n\n    class LastHiddenState(torch.nn.Module):\n        def __init__(self, model):\n            super().__init__()\n            self.model = model\n\n        def forward(self, input_ids, attention_mask, token_type_ids):\n            return self.model(input_ids=input_ids, attention_mask=attention_mask,\n                              token_type_ids=token_type_ids).last_hidden_state\n\n    model = AutoModel.from_pretrained(folder, attn_implementation="eager").eval()\n    sample = AutoTokenizer.from_pretrained(folder)(["query: cook", "passage: a longer text, so the first is padded"],\n                                                   padding=True, return_tensors="pt")\n    inputs = (sample["input_ids"], sample["attention_mask"], torch.zeros_like(sample["input_ids"]))\n    names = ["input_ids", "attention_mask", "token_type_ids"]\n    path = folder / "onnx" / "model.onnx"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with torch.no_grad():\n        torch.onnx.export(LastHiddenState(model), inputs, str(path), input_names=names,\n                          output_names=["last_hidden_state"], opset_version=14, dynamo=False,\n                          dynamic_axes={name: {0: "batch_size", 1: "sequence_length"}\n                                        for name in [*names, "last_hidden_state"]})\n    return path\n\n\ndef parity(model, folder: Path, texts: list[str]) -> float:\n    """Lowest cosine between fastembed\'s vectors for the exported folder (what the API runs) and the model\'s own."""\n    from fastembed import TextEmbedding\n    from fastembed.common.model_description import ModelSource, PoolingType\n\n    name = "tesda-track/" + folder.name\n    if not any(m["model"] == name for m in TextEmbedding.list_supported_models()):\n        TextEmbedding.add_custom_model(model=name, pooling=PoolingType.MEAN, normalization=True,\n                                       sources=ModelSource(hf=name), dim=384, model_file="onnx/model.onnx")\n    served = np.asarray(list(TextEmbedding(name, specific_model_path=str(folder)).embed(texts)), dtype=np.float64)\n    reference = np.asarray(model.encode(texts, normalize_embeddings=True), dtype=np.float64)\n    return float(np.min(np.sum(served * reference, axis=1)))\n\n\ndef main(argv=None) -> None:\n    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    parser.add_argument("--data", type=Path, required=True, help="dataset from `python -m training.v6 --dataset`")\n    parser.add_argument("--base", type=Path, required=True, help="sentence-transformers folder to continue from")\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--epochs", type=float, default=1.0)\n    parser.add_argument("--batch-size", type=int, default=48)\n    parser.add_argument("--chunk", type=int, default=8, help="texts per forward pass; lower it if memory runs out")\n    parser.add_argument("--train-word-embeddings", action="store_true",\n                        help="also update the 250k-token embedding table (needs ~1.1 GB more GPU memory)")\n    parser.add_argument("--hard-k", type=int, default=2, help="hardest bank negatives per query that get gradients")\n    parser.add_argument("--max-passages", type=int, default=96)\n    parser.add_argument("--unrelated-per-batch", type=int, default=8,\n                        help="unrelated goals (no owner) added to every batch, cycling through them")\n    parser.add_argument("--unrelated-weight", type=float, default=1.0)\n    parser.add_argument("--scale", type=float, default=20.0)\n    parser.add_argument("--lr", type=float, default=1e-5)\n    parser.add_argument("--refresh-steps", type=int, default=100)\n    parser.add_argument("--save-steps", type=int, default=100)\n    parser.add_argument("--max-steps", type=int, default=None)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--precision", choices=("auto", "bf16", "fp16", "fp32"), default="auto",\n                        help="auto: bf16 on Ampere or newer, fp16 with a gradient scaler on older GPUs (T4)")\n    parser.add_argument("--resume", action="store_true")\n    args = parser.parse_args(argv)\n\n    import torch\n    from sentence_transformers import SentenceTransformer\n\n    data, texts, positives, neutral, held = load_dataset(args.data)\n    output = args.output.resolve()\n    checkpoint = output / "checkpoint"\n    if (output / "onnx" / "model.onnx").exists() or (output / "training.json").exists():\n        raise SystemExit(f"{output} already holds a finished model: choose a new folder")\n    if checkpoint.exists() and not args.resume:\n        raise SystemExit(f"{checkpoint} holds an unfinished run: add --resume or delete the folder")\n    # chunk changes memory use, not the result, so a resume may lower it.\n    settings = {k: v for k, v in vars(args).items() if k not in ("data", "base", "output", "resume", "chunk")}\n    signature = {"settings": settings, "dataset_sha256": digest(args.data)}\n    output.mkdir(parents=True, exist_ok=True)\n    if args.resume and json.loads((output / "run_config.json").read_text(encoding="utf-8")) != signature:\n        raise SystemExit("Resume data/settings differ from run_config.json")\n    (output / "run_config.json").write_text(json.dumps(signature, indent=2) + "\\n", encoding="utf-8")\n\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    device = "cuda" if torch.cuda.is_available() and torch.cuda.device_count() else "cpu"\n    resuming = args.resume and (checkpoint / "trainer_state.pt").exists()\n    model = SentenceTransformer(str(checkpoint if resuming else args.base), device=device)\n    model.max_seq_length = 512\n    model.eval()  # no dropout: the cached and the re-run forward passes must agree\n    if not args.train_word_embeddings:\n        # Keeps the multilingual vocabulary as the parent had it and frees the optimizer state of 96M parameters.\n        model[0].auto_model.embeddings.word_embeddings.weight.requires_grad_(False)\n    trainable = [p for p in model.parameters() if p.requires_grad]\n    trainer = Trainer(model, data["passages"], args.chunk, args.scale, device, args.precision)\n    to_device = lambda a: torch.from_numpy(a).to(device)  # noqa: E731\n    unrelated = ~positives.any(1)\n    train_idx, held_idx = np.flatnonzero(~held & ~unrelated), np.flatnonzero(held)\n    unrelated_idx = np.flatnonzero(~held & unrelated)\n    if args.unrelated_per_batch and not len(unrelated_idx):\n        raise SystemExit("The dataset has no unrelated goals; rebuild it or pass --unrelated-per-batch 0")\n    held_texts = [texts[i] for i in held_idx]\n    held_pos, held_neu = to_device(positives[held_idx]), to_device(neutral[held_idx])\n\n    steps_per_epoch = math.ceil(len(train_idx) / args.batch_size)\n    total = args.max_steps or math.ceil(steps_per_epoch * args.epochs)\n    optimizer = torch.optim.AdamW(trainable, lr=args.lr, weight_decay=0.01)\n    warmup = max(1, round(0.05 * total))\n    scheduler = torch.optim.lr_scheduler.LambdaLR(\n        optimizer, lambda s: min((s + 1) / warmup, max(0.0, (total - s) / max(1, total - warmup))))\n    start_step, before = 0, None\n    if resuming:\n        state = torch.load(checkpoint / "trainer_state.pt", weights_only=True)\n        optimizer.load_state_dict(state["optimizer"])\n        scheduler.load_state_dict(state["scheduler"])\n        trainer.scaler.load_state_dict(state["scaler"])\n        start_step, before = state["step"], state["before"]\n        print(f"Resuming at step {start_step}", flush=True)\n    order, extra, rng = [], [], random.Random(args.seed)\n    for _ in range(math.ceil(total / steps_per_epoch) + 1):\n        epoch = train_idx.tolist()\n        rng.shuffle(epoch)\n        order += [epoch[i:i + args.batch_size] for i in range(0, len(epoch), args.batch_size)]\n    while len(extra) < total * args.unrelated_per_batch:\n        cycle = unrelated_idx.tolist()\n        rng.shuffle(cycle)\n        extra += cycle\n    order = [batch + extra[step * args.unrelated_per_batch:(step + 1) * args.unrelated_per_batch]\n             for step, batch in enumerate(order[:total])]\n    print(f"{len(texts)} queries ({len(train_idx)} related and {len(unrelated_idx)} unrelated train, "\n          f"{len(held_idx)} held out), {len(data[\'names\'])} passages; {total} steps of {args.batch_size} + "\n          f"{args.unrelated_per_batch} unrelated; device {device} ({trainer.dtype})", flush=True)\n\n    trainer.refresh()\n    if before is None:\n        before = list(trainer.evaluate(held_texts, held_pos, held_neu))\n        print(f"held-out loss before: {before[0]:.4f}; unrelated below z 3.5: {before[1]}", flush=True)\n    started, recent = time.perf_counter(), []\n    for step in range(start_step, total):\n        if step > start_step and step % args.refresh_steps == 0:\n            trainer.refresh()\n        batch = order[step]\n        optimizer.zero_grad(set_to_none=True)\n        loss, used = trainer.step([texts[i] for i in batch], to_device(positives[batch]), to_device(neutral[batch]),\n                                  args.hard_k, args.max_passages, args.unrelated_weight)\n        trainer.scaler.unscale_(optimizer)\n        torch.nn.utils.clip_grad_norm_(trainable, 1.0)\n        trainer.scaler.step(optimizer)\n        trainer.scaler.update()\n        scheduler.step()\n        recent.append(loss)\n        if (step + 1) % 20 == 0 or step + 1 == total:\n            rate = (step + 1 - start_step) / (time.perf_counter() - started)\n            print(f"step {step + 1}/{total} loss {np.mean(recent):.4f} passages {used} "\n                  f"lr {scheduler.get_last_lr()[0]:.2e} {rate:.2f} steps/s eta {(total - step - 1) / rate / 60:.0f} min",\n                  flush=True)\n            recent = []\n        if (step + 1) % args.save_steps == 0 and step + 1 < total:\n            model.save_pretrained(str(checkpoint))\n            torch.save({"optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),\n                        "scaler": trainer.scaler.state_dict(), "step": step + 1, "before": before},\n                       checkpoint / "trainer_state.pt")\n    seconds = time.perf_counter() - started\n    trainer.refresh()\n    after = list(trainer.evaluate(held_texts, held_pos, held_neu))\n    print(f"held-out loss after: {after[0]:.4f}; unrelated below z 3.5: {after[1]}; trained {total - start_step} "\n          f"steps in {seconds / 60:.1f} min", flush=True)\n\n    model.save_pretrained(str(output))\n    export_onnx(output)\n    sample = data["passages"][::max(1, len(data["passages"]) // PARITY_TEXTS)][:PARITY_TEXTS // 2]\n    cosine = parity(model, output, [f"passage: {p}" for p in sample] +\n                    [f"query: {t}" for t in held_texts[:PARITY_TEXTS // 2]])\n    print(f"fastembed parity: lowest cosine {cosine:.6f}", flush=True)\n    if not np.isfinite(cosine) or cosine < MIN_PARITY:\n        raise SystemExit(f"fastembed parity {cosine:.6f} is below {MIN_PARITY}: the ONNX export does not match")\n    recovered = args.base / "recovered_from.txt"\n    metadata = {"profile": data.get("profile", "v6"), "base": args.base.name,\n                "recovered_from": recovered.read_text(encoding="utf-8").splitlines() if recovered.exists() else None,\n                "settings": settings, "dataset_sha256": signature["dataset_sha256"], "data": data["audit"],\n                "steps": total, "held_out_loss": {"before": before[0], "after": after[0]},\n                "held_out_unrelated_below_z35": {"before": before[1], "after": after[1]},\n                "train_seconds": round(seconds, 1), "device": device,\n                "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None, "dtype": str(trainer.dtype),\n                "parity_cosine": cosine, "date": datetime.now(timezone.utc).isoformat(timespec="seconds"),\n                "packages": {n: version(n) for n in ("torch", "sentence-transformers", "transformers", "onnx",\n                                                     "fastembed")}}\n    (output / "training.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n    shutil.rmtree(checkpoint, ignore_errors=True)\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
(WORK / "recover.py").write_text('"""Recover a trainable sentence-transformers folder from a finished model\'s ONNX export.\n\nEarlier rounds kept only the serving files (onnx/model.onnx, tokenizer, config), not model.safetensors. The ONNX\ngraph still holds every weight: named initializers keep their PyTorch names, and the unnamed MatMul weights are\nfound through the bias added right after them. The recovered model must reproduce the ONNX output, or nothing is saved.\n\nRun from backend/ inside .venv-train:\n    python -m training.recover --onnx-folder models/tesda-e5-small-v4 --output models/_torch-v4\n"""\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\n\nBASE_MODEL = "intfloat/multilingual-e5-small"\nPREFIX = "model."\nCHECK_TEXTS = ["query: gusto kong maging welder", "query: cook",\n               "passage: Cookery NC II. Prepare appetizers, salads, soups and sauces for hotel and restaurant kitchens.",\n               "query: marunong akong mag-ayos ng sirang aircon at ref", "passage: Plumbing NC I"]\n\n\ndef onnx_state_dict(path: Path) -> dict[str, np.ndarray]:\n    """PyTorch parameter name -> array, from the initializers of the TorchScript ONNX export."""\n    import onnx\n    from onnx import numpy_helper\n\n    graph = onnx.load(str(path)).graph\n    values = {t.name: numpy_helper.to_array(t) for t in graph.initializer}\n    consumers: dict[str, list] = {}\n    for node in graph.node:\n        for name in node.input:\n            consumers.setdefault(name, []).append(node)\n    state = {}\n    for name, array in values.items():\n        if not name.startswith("onnx::"):\n            state[name] = array\n            continue\n        matmuls = [n for n in consumers.get(name, []) if n.op_type == "MatMul"]\n        if len(matmuls) != 1:\n            raise ValueError(f"{name}: expected one MatMul consumer, found {len(matmuls)}")\n        adds = [n for n in consumers.get(matmuls[0].output[0], []) if n.op_type == "Add"]\n        biases = [i for n in adds for i in n.input if i.endswith(".bias") and i in values]\n        if len(biases) != 1:\n            raise ValueError(f"{name}: cannot find the bias that follows its MatMul")\n        weight = biases[0][: -len(".bias")] + ".weight"\n        if weight in state:\n            raise ValueError(f"{weight} recovered twice")\n        state[weight] = array.T.copy()  # ONNX MatMul multiplies by W^T\n    return {k[len(PREFIX):] if k.startswith(PREFIX) else k: v for k, v in state.items()}\n\n\ndef onnx_outputs(path: Path, tokenizer, texts: list[str]) -> np.ndarray:\n    import onnxruntime as ort\n\n    session = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])\n    batch = tokenizer(texts, padding=True, return_tensors="np")\n    feeds = {"input_ids": batch["input_ids"].astype(np.int64),\n             "attention_mask": batch["attention_mask"].astype(np.int64),\n             "token_type_ids": np.zeros_like(batch["input_ids"], dtype=np.int64)}\n    hidden = session.run(None, feeds)[0]\n    mask = batch["attention_mask"][..., None].astype(np.float64)\n    pooled = (hidden * mask).sum(1) / mask.sum(1)\n    return pooled / np.linalg.norm(pooled, axis=1, keepdims=True)\n\n\ndef recover(onnx_folder: Path, output: Path) -> float:\n    import torch\n    from sentence_transformers import SentenceTransformer\n\n    if output.exists():\n        raise SystemExit(f"{output} exists: choose a new folder")\n    state = onnx_state_dict(onnx_folder / "onnx" / "model.onnx")\n    model = SentenceTransformer(BASE_MODEL, device="cpu")\n    bert = model[0].auto_model\n    current = bert.state_dict()\n    tensors = {k: torch.from_numpy(v) for k, v in state.items() if k in current}\n    unknown = sorted(set(state) - set(current))\n    missing = sorted(k for k in set(current) - set(tensors) if not k.startswith("pooler."))\n    if unknown or missing:\n        raise SystemExit(f"Weight names do not line up. unknown={unknown[:5]} missing={missing[:5]}")\n    for key, tensor in tensors.items():\n        if tuple(tensor.shape) != tuple(current[key].shape):\n            raise SystemExit(f"{key}: shape {tuple(tensor.shape)} != {tuple(current[key].shape)}")\n    bert.load_state_dict(tensors, strict=False)  # the pooler is not in the export and is never used\n    reference = onnx_outputs(onnx_folder / "onnx" / "model.onnx", model.tokenizer, CHECK_TEXTS)\n    ours = model.encode(CHECK_TEXTS, normalize_embeddings=True, convert_to_numpy=True).astype(np.float64)\n    cosine = float(np.min(np.sum(reference * ours, axis=1)))\n    print(f"Recovered {len(tensors)} tensors; lowest cosine against the ONNX export: {cosine:.7f}")\n    if not np.isfinite(cosine) or cosine < 0.9999:\n        raise SystemExit("Recovered weights do not reproduce the ONNX export; nothing saved")\n    model.save_pretrained(str(output))\n    (output / "recovered_from.txt").write_text(f"{onnx_folder.resolve()}\\nmin_cosine={cosine:.7f}\\n", encoding="utf-8")\n    return cosine\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    parser.add_argument("--onnx-folder", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    args = parser.parse_args()\n    recover(args.onnx_folder, args.output)\n', encoding="utf-8")
print("dataset sha256", hashlib.sha256((WORK / "v6-dataset.json").read_bytes()).hexdigest())


In [ ]:
#@title 4. Fetch the published v4 parent, check it and recover trainable weights
import hashlib, shutil, subprocess, sys
from pathlib import Path
from huggingface_hub import snapshot_download
WORK = Path("/content/tesda-v6")
for needed in ("v6-dataset.json", "v6_train.py", "recover.py"):
    assert (WORK / needed).is_file(), "Run cell 3 (Write the embedded training data and code) first"
parent = WORK / "tesda-e5-small-v4"
snapshot_download("Nyanta19/tesda-e5-small-v4", local_dir=parent,
                  allow_patterns=["config.json", "tokenizer.json", "tokenizer_config.json", "onnx/model.onnx"])
with open(parent / "onnx/model.onnx", "rb") as f:
    assert hashlib.file_digest(f, "sha256").hexdigest() == "76cbab6ba7b27672b1f3b5e0b48260b5a4bc0d5d942d4ccaeb625948ab2e5600", "Unexpected v4 parent ONNX"
BASE = WORK / "torch-v4"
if not (BASE / "recovered_from.txt").exists():
    shutil.rmtree(BASE, ignore_errors=True)
    subprocess.run([sys.executable, "-u", str(WORK / "recover.py"), "--onnx-folder", str(parent),
                    "--output", str(BASE)], cwd=WORK, check=True)
print((BASE / "recovered_from.txt").read_text())


In [ ]:
#@title 5. Train v6 (resumes from Drive after a disconnect)
import json, subprocess, sys
from pathlib import Path
WORK = Path("/content/tesda-v6")
assert (WORK / "settings.json").is_file(), "Run cell 2 (Settings) first"
assert (WORK / "torch-v4" / "recovered_from.txt").is_file(), "Run cells 3 and 4 first"
S = json.loads((WORK / "settings.json").read_text())
if S["USE_DRIVE"]:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS = Path("/content/drive/MyDrive/tesda-track-training")
else:
    RUNS = WORK / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
OUT = RUNS / (f"tesda-e5-small-{S['ROUND']}" + ("-smoke" if S["SMOKE"] else ""))
args = [sys.executable, "-u", str(WORK / "v6_train.py"), "--data", str(WORK / "v6-dataset.json"),
        "--base", str(WORK / "torch-v4"), "--output", str(OUT), "--epochs", str(S["EPOCHS"]), "--lr", str(S["LR"]),
        "--batch-size", str(S["BATCH_SIZE"]), "--unrelated-per-batch", str(S["UNRELATED_PER_BATCH"]),
        "--chunk", str(S["CHUNK"])]
if S["TRAIN_WORD_EMBEDDINGS"]: args.append("--train-word-embeddings")
if S["SMOKE"]: args += ["--max-steps", "30", "--save-steps", "20"]
if (OUT / "training.json").exists():
    print("Already trained:", OUT)
else:
    if (OUT / "checkpoint").exists(): args.append("--resume")
    run = subprocess.Popen(args, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in run.stdout:
        print(line, end="")
    assert run.wait() == 0, "Training failed: read the output above"


In [ ]:
#@title 6. Package the serving files with checksums and download them
import hashlib, json, zipfile
from pathlib import Path
from google.colab import files
WORK = Path("/content/tesda-v6")
assert (WORK / "settings.json").is_file(), "Run cell 2 (Settings) first"
S = json.loads((WORK / "settings.json").read_text())
if S["USE_DRIVE"]:
    from google.colab import drive
    drive.mount("/content/drive")
RUNS = Path("/content/drive/MyDrive/tesda-track-training") if S["USE_DRIVE"] else WORK / "runs"
OUT = RUNS / (f"tesda-e5-small-{S['ROUND']}" + ("-smoke" if S["SMOKE"] else ""))
assert (OUT / "training.json").is_file(), f"{OUT} has no finished model: run cell 5 (Train) first"
serving = ['config.json', 'tokenizer.json', 'tokenizer_config.json', 'training.json', 'run_config.json', 'modules.json', 'sentence_bert_config.json', 'config_sentence_transformers.json', '1_Pooling/config.json', 'onnx/model.onnx']
missing = [p for p in serving if not (OUT / p).is_file()]
assert not missing, f"Missing serving files: {missing}"
checksums = {}
for p in serving:
    with open(OUT / p, "rb") as f:
        checksums[p] = hashlib.file_digest(f, "sha256").hexdigest()
archive = Path("/content") / f"{OUT.name}.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in serving:
        z.write(OUT / p, p)
    z.writestr("2_Normalize/.keep", "")
    z.writestr("checksums.json", json.dumps(checksums, indent=2))
# Check the archive before downloading: a truncated model.onnx would still have a valid size in Drive.
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    assert hashlib.sha256(z.read("onnx/model.onnx")).hexdigest() == checksums["onnx/model.onnx"]
meta = json.loads((OUT / "training.json").read_text())
print(json.dumps({k: meta[k] for k in ("held_out_loss", "held_out_unrelated_below_z35")}))
print("Downloading", archive, archive.stat().st_size // 2**20, "MB")
files.download(str(archive))
